### LogSumEXp trick

In [ ]:
Y_hat = Y_hat.reshape((-1, Y_hat.shape[-1]))
Y = Y.reshape((-1,))

Typical shapes in classification:
- `Y_hat (logits)` often comes as `(batch_size, num_classes)` or for sequences/images maybe `(batch_size, time, num_classes)` or `(batch_size, H, W, num_classes)` depending on how they arranged it.
- `Y` (labels) matches the "non-class" dimensions: `(batch_size, )` or `(batch_size, time)` etc.

PyTorch's `F.cross_entropy` expects:
- input: `(N, C)` where `N` is "number of examples" and `C` is classes
- target: `(N, )` with integer class indices

So these reshapes flatten everything except the class dimension into one big N .

Example:
- If `Y_hat` is `(batch =32, time =10, classes =5 )` , after reshape it becomes `(320,5)`
- If `Y` is `(32,10)`, after reshape it becomes `(320,)`

In [ ]:
return F.cross_entropy(Y_hat, Y, reduction='mean' if averaged else 'none')

That's precisely the math in the note:
- subtract $\bar{o}$ (a max) to keep exponentials safe
- combine softmax $+\log$ together, so you never take $\log (0)$ from underflowed probabilities

So the mapping is:
- Note: "pass logits into loss; compute softmax+log stably inside"
- Code: `F.cross_entropy(Y_hat, Y, ...)` where `Y_hat` is logits

In [ ]:
reduction='mean' if averaged else 'none'

- mean : return a single scalar = average loss over all N flattened examples
- none : return per-example losses (shape `( N, ) `) so you could weight them or inspect them

Risky way (what the notes say to avoid):

In [ ]:
probs = softmax(logits)          # can under/overflow
loss = -torch.log(probs[range(N), y]).mean()  # log(0) -> -inf -> NaN in backward

Safe way:

In [ ]:
loss = F.cross_entropy(logits, y)  # internally logsumexp + log_softmax